In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic' 
plt.rcParams['axes.unicode_minus'] = False

In [10]:
df = pd.read_csv('../../data/DieCasting_Quality_Raw_Data.csv',header=[0,1])
df

Process                                                     \
           id Product_Type Shot Velocity_1 Velocity_2 Velocity_3   
0           1            1    1      0.144      0.170      0.188   
1        1002            1    2      0.144      0.170      0.182   
2        2003            1    3      0.144      0.170      0.182   
3        3004            1    4      0.144      0.170      0.182   
4        4005            1    5      0.144      0.172      0.176   
...       ...          ...  ...        ...        ...        ...   
7530  7530659            2  659      0.150      0.166      0.210   
7531  7531660            2  660      0.144      0.174      0.206   
7532  7532660            2  660      0.144      0.174      0.206   
7533  7533661            2  661      0.147      0.174      0.204   
7534  7534661            2  661      0.147      0.174      0.204   

                                                                         ...  \
     High_Velocity Cylinder_Pressure Rapid_Rise_Time Biscuit_Thickness   ...   
0            2.134               214           0.008                 10  ...   
1            2.124               217           0.008                 11  ...   
2            2.116               214           0.008                 11  ...   
3            2.137               217           0.008                 11  ...   
4            2.111               217           0.008                 12  ...   
...            ...               ...             ...                ...  ...   
7530         2.492               265           0.011                 17  ...   
7531         2.514               264           0.011                 16  ...   
7532         2.514               264           0.011                 16  ...   
7533         2.532               265           0.012                 18  ...   
7534         2.532               265           0.012                 18  ...   

         Defects                                                          \
     Blow_Hole_2 Stain_2 Dent_2 Deformation_2 Contamination_2 Impurity_2   
0              0       0      0             0               0          0   
1              0       0      0             0               0          0   
2              0       0      0             0               0          0   
3              0       0      0             0               0          0   
4              0       0      0             0               0          0   
...          ...     ...    ...           ...             ...        ...   
7530           0       0      0             0               0          0   
7531           0       0      0             0               0          0   
7532           0       0      0             0               0          0   
7533           0       0      0             0               0          0   
7534           0       0      0             0               0          0   

                                                   
     Crack_2 Scratch_2 Buring_Mark_2 Inclusions_2  
0          0         0             0            0  
1          0         0             0            0  
2          0         0             0            0  
3          0         0             0            0  
4          0         0             0            0  
...      ...       ...           ...          ...  
7530       0         0             0            0  
7531       0         0             0            0  
7532       0         0             0            0  
7533       0         0             0            0  
7534       0         0             0            0  

[7535 rows x 57 columns]

1. 컬럼명 공백제거
2. 중복제거
3. 2번, 3번을 불량(1)로 대체
4. 새로운 파생컬럼 생성

In [11]:
### 1. 컬럼명 공백 제거
df.rename(columns={'Biscuit_Thickness ': 'Biscuit_Thickness', 'Clamping_Force ': 'Clamping_Force', ' Pressure_Rise_Time': 'Pressure_Rise_Time'}, level=1, inplace=True)

In [12]:
### 2. 중복 제거
# 제외할 컬럼 리스트 정의
exclude_cols = [('Process', 'id')]

# 전체 컬럼에서 제외할 컬럼만 뺀 리스트 생성
target_cols = df.columns.difference(exclude_cols).tolist()

# 중복 제거 실행 (첫 번째 행은 남기고 나머지는 삭제)
df_cleaned = df.drop_duplicates(subset=target_cols, keep='first').reset_index(drop=True)

# 결과 확인
print(f"제거 전 행 개수: {len(df)}")
print(f"제거 후 행 개수: {len(df_cleaned)}")
print(f"삭제된 중복 행 개수: {len(df) - len(df_cleaned)}")

제거 전 행 개수: 7535
제거 후 행 개수: 4617
삭제된 중복 행 개수: 2918


In [13]:
### 3. 2,3을 불량 1로 대체
defect_cols = df_cleaned['Defects'].columns

# 'Defects' 대분류 아래의 모든 컬럼에 대해 2 이상의 값을 1로 변경
for col in defect_cols:
    # Defects 하위의 col 값 중 2 이상인 경우를 1로 치환
    df_cleaned.loc[df_cleaned[('Defects', col)] >= 2, ('Defects', col)] = 1

In [14]:
### 4. 새로운 파생컬럼 생성
defect_cols = [col for col in df_cleaned.columns if col[0]=='Defects']

df_cleaned[('Defect_Flag','Is_Defect')] = (df_cleaned[defect_cols] == 1).any(axis=1).astype(int)

df_cleaned[('Defect_Flag','Is_Defect')].value_counts() # --> 0: 정상, 1: 불량

(Defect_Flag, Is_Defect)
0    3544
1    1073
Name: count, dtype: int64

In [15]:
df_cleaned

Process                                                     \
           id Product_Type Shot Velocity_1 Velocity_2 Velocity_3   
0           1            1    1      0.144      0.170      0.188   
1        1002            1    2      0.144      0.170      0.182   
2        2003            1    3      0.144      0.170      0.182   
3        3004            1    4      0.144      0.170      0.182   
4        4005            1    5      0.144      0.172      0.176   
...       ...          ...  ...        ...        ...        ...   
4612  7525657            2  657      0.144      0.173      0.200   
4613  7527658            2  658      0.144      0.173      0.200   
4614  7529659            2  659      0.150      0.166      0.210   
4615  7531660            2  660      0.144      0.174      0.206   
4616  7533661            2  661      0.147      0.174      0.204   

                                                                        ...  \
     High_Velocity Cylinder_Pressure Rapid_Rise_Time Biscuit_Thickness  ...   
0            2.134               214           0.008                10  ...   
1            2.124               217           0.008                11  ...   
2            2.116               214           0.008                11  ...   
3            2.137               217           0.008                11  ...   
4            2.111               217           0.008                12  ...   
...            ...               ...             ...               ...  ...   
4612         2.536               264           0.012                17  ...   
4613         2.536               264           0.012                17  ...   
4614         2.492               265           0.011                17  ...   
4615         2.514               264           0.011                16  ...   
4616         2.532               265           0.012                18  ...   

     Defects                                                          \
     Stain_2 Dent_2 Deformation_2 Contamination_2 Impurity_2 Crack_2   
0          0      0             0               0          0       0   
1          0      0             0               0          0       0   
2          0      0             0               0          0       0   
3          0      0             0               0          0       0   
4          0      0             0               0          0       0   
...      ...    ...           ...             ...        ...     ...   
4612       0      0             0               0          0       0   
4613       0      0             0               0          0       0   
4614       0      0             0               0          0       0   
4615       0      0             0               0          0       0   
4616       0      0             0               0          0       0   

                                          Defect_Flag  
     Scratch_2 Buring_Mark_2 Inclusions_2   Is_Defect  
0            0             0            0           0  
1            0             0            0           0  
2            0             0            0           0  
3            0             0            0           1  
4            0             0            0           0  
...        ...           ...          ...         ...  
4612         0             0            0           0  
4613         0             0            0           0  
4614         0             0            0           0  
4615         0             0            0           1  
4616         0             0            0           1  

[4617 rows x 58 columns]

In [16]:
# 1. Product_Type이 1인 행만 필터링하여 저장
df_type_1 = df_cleaned[df_cleaned[('Process', 'Product_Type')] == 1].reset_index(drop=True)
df_type_1.to_csv('product_type_1.csv', index=False, encoding='utf-8-sig')

# 2. Product_Type이 2인 행만 필터링하여 저장
df_type_2 = df_cleaned[df_cleaned[('Process', 'Product_Type')] == 2].reset_index(drop=True)
df_type_2.to_csv('product_type_2.csv', index=False, encoding='utf-8-sig')

In [17]:
df1 = pd.read_csv('product_type_1.csv',header=[0,1])
df2 = pd.read_csv('product_type_2.csv',header=[0,1])

In [18]:
# 컬럼별 결측치 개수 합계
print(df1.isnull().sum())

Process      id                      0
             Product_Type            0
             Shot                    0
             Velocity_1              0
             Velocity_2              0
             Velocity_3              0
             High_Velocity           0
             Cylinder_Pressure       0
             Rapid_Rise_Time         0
             Biscuit_Thickness       0
             Clamping_Force          0
             Cycle_Time              0
             Pressure_Rise_Time      0
             Casting_Pressure        0
             Spray_Time              0
             Spray_1_Time            0
             Spray_2_Time            0
Sensor       Melting_Furnace_Temp    0
             Air_Pressure            0
             Air_Pressure_Min        0
             Air_Pressure_Max        0
             Coolant_Temp            0
             Coolant_Temp_Min        0
             Coolant_Temp_Max        0
             Coolant_Pressure        0
             Factory_Temp

In [19]:
# 컬럼별 결측치 개수 합계
print(df2.isnull().sum())

Process      id                       0
             Product_Type             0
             Shot                     0
             Velocity_1               0
             Velocity_2               0
             Velocity_3               0
             High_Velocity            0
             Cylinder_Pressure        0
             Rapid_Rise_Time          0
             Biscuit_Thickness        0
             Clamping_Force           0
             Cycle_Time               0
             Pressure_Rise_Time       0
             Casting_Pressure         0
             Spray_Time               0
             Spray_1_Time             0
             Spray_2_Time             0
Sensor       Melting_Furnace_Temp     0
             Air_Pressure             0
             Air_Pressure_Min         0
             Air_Pressure_Max         0
             Coolant_Temp             0
             Coolant_Temp_Min         0
             Coolant_Temp_Max         0
             Coolant_Pressure         0



1. 데이터부터 확인
2. describe-결측치 확인
3. train / test 분리 => 테스트 =? 3식


4. 결측치 데이터를 어떻게할지